# Session 2 — Blood Cell Classification · Data Analysis & Problem Understanding

*Personal EDA companion to `blood_cell_classification.ipynb`. **It does not submit** — it
downloads the images and inspects them so you understand the classification problem before
training.*

**The task.** Classify a **28×28 RGB** microscopy image of a blood cell into one of
**8 cell types**. Scored on **F1-macro** — every class counts equally, so the *rare* cell
types matter as much as the common ones.

What this notebook answers:
1. How many images, what do they look like, what's the per-class balance?
2. What does the trivial F1-macro baseline score (so you know the floor)?
3. What visually distinguishes the classes (mean images, colour, brightness)?
4. Are classes separable in raw pixel space — and which ones get confused?
5. Any data-quality issues (duplicates, degenerate images, train/test shift)?

## 0. Setup

In [ ]:
!pip -q install "mlarena-sdk==0.3.0" seaborn   # installs the `mlarena` package + plotting

import os, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid", context="notebook")

import mlarena
API_TOKEN = "mlk_user_REPLACE_ME"   # <-- paste your token (Profile page on ml-arena.com)
COMPETITION_ID = 173
client = mlarena.connect(api_key=API_TOKEN, base_url="https://ml-arena.com")

## 1. Download & load the data

`download_dataset` pulls the files into `data/`. We skip the download if they're already
there, then line each image up with its label by `image_id` (not by row order).

In [ ]:
if not os.path.exists("data/train_images.npz"):
    client.download_dataset(COMPETITION_ID, "data/")

train = np.load("data/train_images.npz")   # train["image_id"], train["images"] (N,28,28,3) uint8
test  = np.load("data/test_images.npz")
y_df  = pd.read_csv("data/y_train.csv")     # image_id, label

images = train["images"]
y = y_df.set_index("image_id")["label"].loc[train["image_id"]].values
classes = sorted(pd.unique(y))

print("train images:", images.shape, "  test images:", test["images"].shape)
print("dtype:", images.dtype, " value range:", images.min(), "-", images.max())
print(f"{len(classes)} classes: {classes}")

## 2. Class balance & the F1-macro floor

F1-macro averages the per-class F1, so a model that ignores a rare class is punished hard.
The majority-prediction baseline is near-zero F1-macro — that's the floor.

In [ ]:
vc = pd.Series(y).value_counts().sort_index()
print(vc.to_string())
print(f"\nimbalance ratio (max/min) = {vc.max() / vc.min():.1f}")

from sklearn.metrics import f1_score
maj = vc.idxmax()
print(f"F1-macro if you always predict '{maj}' = {f1_score(y, [maj]*len(y), average='macro'):.4f}")

vc.plot.bar(rot=45, color="#4c72b0"); plt.title("Class distribution"); plt.ylabel("images"); plt.show()

## 3. Look at the images

Always look before modelling. A grid of samples per class, then the **per-class mean image**
— the "average" cell, which reveals colour/shape signatures the model can latch onto.

In [ ]:
n_show = 8
fig, axes = plt.subplots(len(classes), n_show, figsize=(1.3*n_show, 1.3*len(classes)))
rng = np.random.RandomState(0)
for r, cl in enumerate(classes):
    idx = np.where(y == cl)[0]; idx = rng.choice(idx, min(n_show, len(idx)), replace=False)
    for k in range(n_show):
        ax = axes[r, k]; ax.set_xticks([]); ax.set_yticks([])
        if k < len(idx): ax.imshow(images[idx[k]])
        else:            ax.axis("off")
    axes[r, 0].set_ylabel(str(cl), rotation=0, ha="right", va="center", fontsize=9)
plt.suptitle("Random samples per class"); plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(13, 6.5))
for ax, cl in zip(axes.ravel(), classes):
    ax.imshow(images[y == cl].mean(0).astype("uint8"))
    ax.set_title(f"{cl}  (n={int((y == cl).sum())})"); ax.axis("off")
for ax in axes.ravel()[len(classes):]: ax.axis("off")
plt.suptitle("Per-class mean image"); plt.tight_layout(); plt.show()

## 4. Colour & brightness signatures

Microscopy stains differ between cell types. Mean RGB per class and the spread of overall
brightness show how much *colour alone* separates the classes (a cheap feature a CNN gets for free).

In [ ]:
chan = pd.DataFrame({cl: images[y == cl].reshape(-1, 3).mean(0) for cl in classes},
                    index=["R", "G", "B"]).T
chan.plot.bar(color=["#c44e52", "#55a868", "#4c72b0"], figsize=(10, 4))
plt.title("Mean channel intensity per class"); plt.ylabel("0–255"); plt.xticks(rotation=45); plt.show()
chan.round(1)

In [ ]:
bright = images.reshape(len(images), -1).mean(1)
sns.boxplot(data=pd.DataFrame({"label": y, "brightness": bright}), x="label", y="brightness")
plt.xticks(rotation=45); plt.title("Mean pixel brightness by class"); plt.show()

## 5. Are the classes separable in pixel space?

A PCA(2) scatter of the raw (flattened) pixels. Tight, separated clusters ⇒ even simple
models do well; heavy overlap ⇒ you'll need a CNN to exploit spatial structure.

In [ ]:
from sklearn.decomposition import PCA
Xf = images.reshape(len(images), -1) / 255.0
sel = np.random.RandomState(0).choice(len(Xf), min(4000, len(Xf)), replace=False)
pc = PCA(2, random_state=0).fit_transform(Xf[sel])
sns.scatterplot(x=pc[:, 0], y=pc[:, 1], hue=y[sel], s=10, palette="tab10")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.title("PCA(2) of raw pixels"); plt.tight_layout(); plt.show()

## 6. Data-quality checks

Exact-duplicate and degenerate (constant) images can inflate validation scores or signal
leakage between train and test.

In [ ]:
import hashlib
flat = images.reshape(len(images), -1)
h_train = pd.Series([hashlib.md5(r.tobytes()).hexdigest() for r in flat])
h_test  = pd.Series([hashlib.md5(r.tobytes()).hexdigest() for r in test["images"].reshape(len(test["images"]), -1)])
print("exact-duplicate train images:", int(h_train.duplicated().sum()))
print("constant (single-colour) train images:", int((flat.std(1) == 0).sum()))
print("train/test identical-image overlap:", len(set(h_train) & set(h_test)))

## 7. A pixel baseline — and which classes confuse it

Reproduce the starter's flatten-pixels random forest on a stratified hold-out, then read
the per-class F1 and the confusion matrix. The off-diagonal tells you *which cell pairs* are
the hard part of the problem — exactly what a CNN needs to fix.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

Xtr, Xva, ytr, yva = train_test_split(Xf, y, test_size=0.2, random_state=0, stratify=y)
rf = RandomForestClassifier(150, n_jobs=-1, random_state=0).fit(Xtr, ytr)
pred = rf.predict(Xva)
print(f"pixel-RF hold-out F1-macro = {f1_score(yva, pred, average='macro'):.4f}\n")
print(classification_report(yva, pred))

In [ ]:
ConfusionMatrixDisplay(confusion_matrix(yva, pred, labels=classes), display_labels=classes)\
    .plot(xticks_rotation=45, cmap="Blues", colorbar=False)
plt.title("Confusion matrix — pixel RF (which classes get mixed up)"); plt.tight_layout(); plt.show()

## 8. Takeaways → modelling roadmap

- **The floor is ~0 F1-macro** (majority guess); the pixel RF clears it but plateaus because
  flattening throws away the 2-D layout. The score lives in the spatial structure.
- **Rare classes drive F1-macro.** Check the per-class F1 above — the lowest one is your
  bottleneck. Stratify your validation split and consider class weights / oversampling.
- **Colour helps but isn't enough** — overlapping classes in the PCA plot and the confusion
  matrix show which pairs need shape, not just stain.
- **Train a small CNN** (a couple of conv+pool blocks beats the RF by a wide margin). Normalize
  to `[0,1]` or per-channel mean/std; watch a held-out F1 every epoch with early stopping.
- **Augment** (flips, small rotations, mild colour jitter) — cheap robustness on microscopy.
- **Transfer learning** (ResNet/EfficientNet pretrained on ImageNet, resize 28×28 up, fine-tune)
  is usually the biggest jump for the least effort.

Back to the competition starter: `blood_cell_classification.ipynb`.